# City Coverage Check

In [13]:
import pickle
from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd
from shapely.ops import unary_union
from tqdm.auto import tqdm

from blocksnet.analysis.land_use.prediction.core import (
    category_to_index,
    land_use_to_category,
    str_to_land_use,
)
from blocksnet.analysis.land_use.prediction.schemas import BlocksInputSchema

ox.settings.use_cache = True


def load_gdfs(root: str | Path, pattern: str = "*.pkl", target_crs: str | None = None) -> list[gpd.GeoDataFrame]:
    """Load GeoDataFrames from pickle files located under ``root``."""
    root_path = Path(root)
    if root_path.is_file() and root_path.suffix.lower() == ".pkl":
        files = [root_path]
    elif root_path.is_dir():
        files = sorted(p for p in root_path.rglob(pattern) if p.suffix.lower() == ".pkl")
    else:
        return []

    def ensure_gdf(obj):
        if isinstance(obj, gpd.GeoDataFrame):
            return obj
        if isinstance(obj, pd.DataFrame) and "geometry" in obj.columns:
            return gpd.GeoDataFrame(obj, geometry="geometry", crs=getattr(obj, "crs", None))
        return None

    def infer_city_country(fp: Path):
        city = fp.stem
        country = fp.parent.name if fp.parent != fp else None
        return city, country

    gdfs: list[gpd.GeoDataFrame] = []
    for fp in files:
        try:
            with open(fp, "rb") as f:
                obj = pickle.load(f)
        except Exception as exc:  # pragma: no cover - logging only
            print(f'Failed to load {fp}: {exc}')
            continue

        gdf = ensure_gdf(obj)
        if gdf is None and isinstance(obj, dict):
            for key, value in obj.items():
                gi = ensure_gdf(value)
                if gi is None:
                    continue
                gi = gi.copy()
                if "city" not in gi.columns or gi["city"].isna().all():
                    gi["city"] = str(key)
                _, country = infer_city_country(fp)
                if "country" not in gi.columns and country:
                    gi["country"] = country
                if target_crs:
                    gi = gi.to_crs(target_crs) if gi.crs else gi.set_crs(target_crs)
                gdfs.append(gi)
            continue

        if gdf is None:
            print(f'Skip {fp}: unsupported payload type')
            continue

        gdf = gdf.copy()
        city, country = infer_city_country(fp)
        if "city" not in gdf.columns:
            gdf["city"] = city
        if "country" not in gdf.columns and country:
            gdf["country"] = country
        if target_crs:
            gdf = gdf.to_crs(target_crs) if gdf.crs else gdf.set_crs(target_crs)
        gdfs.append(gdf)

    return gdfs


def map_and_filter_blocks(blocks: list[gpd.GeoDataFrame]) -> list[gpd.GeoDataFrame]:
    filtered: list[gpd.GeoDataFrame] = []
    for gdf in blocks:
        gi = gdf.copy()
        if "land_use" in gi.columns:
            gi["category"] = gi["land_use"].map(str_to_land_use).map(land_use_to_category)
        if "category" in gi and gi["category"].notna().sum() > 0:
            filtered.append(gi)
    return filtered


def normalize_and_filter(data):
    if isinstance(data, (list, tuple)):
        result = []
        for gdf in data:
            gi = BlocksInputSchema(gdf)
            if "land_use" in gi:
                gi["category"] = gi["land_use"].map(str_to_land_use).map(land_use_to_category)
            gi = gi[gi["category"].map(category_to_index).notna()]
            if not gi.empty:
                gi['city'] = gdf['city'].iloc[0]
                gi['country'] = gdf['country'].iloc[0]
                result.append(gi)
        return result
    gi = BlocksInputSchema(data)
    if "land_use" in gi:
        gi["category"] = gi["land_use"].map(str_to_land_use).map(land_use_to_category)
    return gi[gi["category"].map(category_to_index).notna()]


In [14]:
data_root = Path("data/TANYA-2025/")
blocks_gdf = load_gdfs(data_root)
if isinstance(blocks_gdf, gpd.GeoDataFrame):
    blocks_gdf = [blocks_gdf]
blocks_gdf_copy = normalize_and_filter(map_and_filter_blocks(blocks_gdf))
print(f"Loaded {len(blocks_gdf_copy)} city GeoDataFrames")


Loaded 2126 city GeoDataFrames


In [16]:
blocks_gdf_copy[0]

,geometry,category,city,country
0,"POLYGON ((7695233.28 4347830.673, 7695239.313 ...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
1,"POLYGON ((7695357.312 4345125.413, 7695335.716...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
2,"POLYGON ((7695305.481 4339252.936, 7695237.554...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
3,"POLYGON ((7696979.816 4333980.991, 7696948.112...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
4,"POLYGON ((7646651.829 4297389.817, 7646653.98 ...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
...,...,...,...,...
1860,"POLYGON ((7701818.262 4339531.938, 7701841.55 ...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
1861,"POLYGON ((7680725.868 4333333.253, 7680717.652...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
1862,"POLYGON ((7680704.149 4333363.948, 7680705.697...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan
1863,"POLYGON ((7670397.066 4360671.545, 7670427.278...",LandUseCategory.RESIDENTIAL,Baghlān,Afghanistan


In [ ]:
def choose_area_crs(gdf: gpd.GeoDataFrame, fallback: str = "EPSG:3857") -> str:
    try:
        crs = gdf.estimate_utm_crs()
        if crs:
            authority = getattr(crs, "to_authority", None)
            if authority:
                epsg = authority()
                if epsg:
                    return f"{epsg[0]}:{epsg[1]}"
            return str(crs)
    except Exception:
        pass
    return fallback


def get_city_country(gdf: gpd.GeoDataFrame) -> tuple[str | None, str | None]:
    city = None
    if "city" in gdf.columns:
        values = [v for v in gdf["city"].unique() if isinstance(v, str) and v.strip()]
        if values:
            city = values[0]
    country = None
    if "country" in gdf.columns:
        values = [v for v in gdf["country"].unique() if isinstance(v, str) and v.strip()]
        if values:
            country = values[0]
    return city, country


def geocode_city_boundary(city: str | None, country: str | None) -> gpd.GeoDataFrame | None:
    if not city:
        return None
    query = ", ".join(part for part in (city, country) if part)
    try:
        boundary = ox.geocode_to_gdf(query)
    except Exception as exc:
        print(f"OSM lookup failed for {query}: {exc}")
        return None
    return boundary if not boundary.empty else None


def compute_coverage(city_gdf: gpd.GeoDataFrame) -> dict | None:
    city, country = get_city_country(city_gdf)
    boundary_gdf = geocode_city_boundary(city, country)
    if boundary_gdf is None:
        return None

    crs = choose_area_crs(city_gdf)
    boundary_proj = boundary_gdf.to_crs(crs)
    city_proj = city_gdf.to_crs(crs)

    city_boundary = unary_union(boundary_proj.geometry)
    if city_boundary.is_empty:
        return None

    source_union = unary_union(city_proj.geometry)
    if source_union.is_empty:
        return None

    covered_area = source_union.intersection(city_boundary).area
    source_area = source_union.area
    city_area = city_boundary.area
    coverage_pct = 0.0 if city_area == 0 else (covered_area / city_area) * 100

    return {
        "city": city or "<unknown>",
        "country": country,
        "city_area_km2": city_area / 1_000_000,
        "covered_area_km2": covered_area / 1_000_000,
        "source_area_km2": source_area / 1_000_000,
        "coverage_pct": coverage_pct,
    }


results = []
for gdf in tqdm(blocks_gdf_copy, desc="Cities"):
    metrics = compute_coverage(gdf)
    if metrics is None:
        continue
    results.append(metrics)

coverage_df = pd.DataFrame(results)
coverage_df


Cities:   1%|          | 14/2126 [03:49<30:57:38, 52.77s/it]

OSM lookup failed for Annaba, Algeria: ('Connection broken: IncompleteRead(16171 bytes read, 297196 more expected)', IncompleteRead(16171 bytes read, 297196 more expected))


Cities:   3%|▎         | 67/2126 [06:24<1:35:24,  2.78s/it] 

In [ ]:
ax = coverage_df["coverage_pct"].plot.hist(bins=20, figsize=(8, 4))
ax.set_xlabel("% покрытия городской территории")
ax.set_ylabel("Количество городов")
ax.set_title("Распределение покрытия блоками")
ax.grid(True, linestyle="--", alpha=0.5)
